# LLM-Based GORE Pipeline — Execution

This notebook executes the complete Goal-Oriented Requirements Engineering pipeline. It separates the initial top-down generation from the proposed iterative bottom-up extension, while reusing the same evaluated low-level-goal generator whenever a decomposition must be created again.

The execution flow is:

1. environment and dataset configuration;
2. evaluated top-down extraction of actors, high-level goals, and low-level goals;
3. bounded bottom-up reconstruction and global evaluation;
4. selective top-down regeneration of low-level goals when a branch changes or a missing high-level goal is added;
5. restart of bottom-up reconstruction and re-evaluation on the updated hierarchy;
6. persistence of all intermediate and final artifacts;
7. goal-to-API alignment using the final verified low-level goals.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from key import (
    get_key_openai,
    get_key_llama,
    count_Llama_keys,
)

openai_key = get_key_openai()
groq_key = get_key_llama()

print("Chiave OpenAI presente:", bool(openai_key))
print("Formato OpenAI plausibile:", openai_key.startswith("sk-"))

print("Chiave Groq presente:", bool(groq_key))
print("Formato Groq plausibile:", groq_key.startswith("gsk_"))

print("Numero chiavi Groq:", count_Llama_keys())

Project root: c:\Users\agnes_lryeu3v\Desktop\tesi
Chiave OpenAI presente: True
Formato OpenAI plausibile: True
Chiave Groq presente: True
Formato Groq plausibile: True
Numero chiavi Groq: 5


## 1. Environment and configuration

This section configures the project path, imports the pipeline components, selects the prompting mode, enables or disables the LLaMA ablation, and loads the datasets used during execution.


In [2]:
from pathlib import Path
import time
import json
import os
import sys

# Resolve the project root whether the notebook is launched from the project
# directory or from a notebooks/ subdirectory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from groundtruth import (
    GENOME,
    GESTAO_HOSPITAL,
    SIA_PROJECT_25_26,
    LONDON_AMBULANCE_SYSTEM,
)
from src.extraction.extractor import (
    generate_description,
    generate_actors,
    generate_high_level_goals,
    generate_low_level_goals,
)
from src.mapping.APIs_mapping import (
    generate_mapping_apis_goals,
    print_api_goal_mapping,
)
from src.self_critique.refine_response import (
    EvalMode,
    generate_response_with_reflection,
)
from src.utils import get_api_list_from_swagger
from src.examples.shot_learning import ShotPromptingMode
#GENOME,
#GESTAO_HOSPITAL,
#LONDON_AMBULANCE_SYSTEM,

GROUNDTRUTHS = [
    SIA_PROJECT_25_26,
]

LLAMA_ABLATION = False
PROMPTING_MODE = ShotPromptingMode.FEW_SHOT
OUTPUT_PATH = PROJECT_ROOT / "output"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)


def output_file_for(dataset_name: str) -> Path:
    suffix = "_noLlama" if LLAMA_ABLATION else ""
    return OUTPUT_PATH / f"{dataset_name}_{PROMPTING_MODE.name}{suffix}.json"


## 2. Baseline top-down pipeline

The baseline architecture follows a top-down process. Starting from the textual documentation, it extracts actors, identifies high-level goals associated with those actors, and decomposes each high-level goal into operational low-level goals. Each extraction stage is executed through the existing reflection-based refinement mechanism.

The original architecture is kept conceptually separate from the proposed extension so that its outputs can be reused as a baseline during the experimental evaluation.


### 2.1 Actor extraction

The actor extractor identifies the entities that interact with the system or pursue goals through it. The reflection loop evaluates and, when required, refines the generated actor set.

### 2.2 High-level goal extraction

The high-level goal extractor derives abstract stakeholder or system intentions from the documentation and associates them with the previously extracted actors.

### 2.3 Low-level goal extraction

The low-level goal extractor decomposes the high-level goals into concrete and operational objectives. These definitive low-level goals become the input of the proposed bottom-up reconstruction stage.


In [9]:
generated_descriptions = {}
generated_actors = {}
generated_hl = {}
generated_ll = {}
generated_actor_objects = {}
generated_hl_objects = {}
generated_ll_objects = {}
baseline_errors = {}


def generate_evaluated_low_level_goals(
    high_level_goals,
    description,
    actors,
):
    """Run the existing top-down LLG generator with its evaluator.

    The same function is used both for the initial decomposition and for every
    selective regeneration requested by the global feedback cycle.
    """
    low_level_goals, score, critique = generate_response_with_reflection(
        "Low Level Goals",
        generate_low_level_goals,
        define_args=(high_level_goals,),
        eval_mode=EvalMode.LOW_LEVEL,
        eval_args=(description, actors, high_level_goals),
        shotPromptingMode=PROMPTING_MODE,
        llama_ablation=LLAMA_ABLATION,
    )
    return low_level_goals, score, critique


def run_baseline_for_dataset(groundtruth: dict) -> None:
    dataset_name = groundtruth["name"]

    try:
        description = (
            str(generate_description(groundtruth["link-readme"]))
            if "link-readme" in groundtruth
            else groundtruth["description"]
        )
        generated_descriptions[dataset_name] = description

        actors, actors_score, actors_critique = generate_response_with_reflection(
            "Actors",
            generate_actors,
            define_args=(description,),
            eval_mode=EvalMode.ACTORS,
            eval_args=(description,),
            shotPromptingMode=PROMPTING_MODE,
            llama_ablation=LLAMA_ABLATION,
        )

        generated_actor_objects[dataset_name] = actors
        generated_actors[dataset_name] = [actor.name for actor in actors.actors]

        high_level_goals, hl_score, hl_critique = generate_response_with_reflection(
            "High Level Goals",
            generate_high_level_goals,
            define_args=(description, actors),
            eval_mode=EvalMode.HIGH_LEVEL,
            eval_args=(description, actors),
            shotPromptingMode=PROMPTING_MODE,
            llama_ablation=LLAMA_ABLATION,
        )

        generated_hl_objects[dataset_name] = high_level_goals
        generated_hl[dataset_name] = [
            goal.description for goal in high_level_goals.goals
        ]

        low_level_goals, ll_score, ll_critique = (
            generate_evaluated_low_level_goals(
                high_level_goals=high_level_goals,
                description=description,
                actors=actors,
            )
        )

        generated_ll_objects[dataset_name] = low_level_goals
        generated_ll[dataset_name] = [
            goal.description for goal in low_level_goals.low_level_goals
        ]

        output = {
            "name": dataset_name,
            "description": description,
            "actors": generated_actors[dataset_name],
            "highLevelGoals": generated_hl[dataset_name],
            "lowLevelGoals": generated_ll[dataset_name],
            "baselineScores": {
                "actors": actors_score,
                "highLevelGoals": hl_score,
                "lowLevelGoals": ll_score,
            },
            "baselineCritiques": {
                "actors": actors_critique,
                "highLevelGoals": hl_critique,
                "lowLevelGoals": ll_critique,
            },
        }

        with output_file_for(dataset_name).open("w", encoding="utf-8") as file:
            json.dump(output, file, indent=4, ensure_ascii=False, default=str)

        print(f"[{dataset_name}] Baseline extraction completed.")

    except Exception as error:
        baseline_errors[dataset_name] = f"{type(error).__name__}: {error}"
        print(f"[{dataset_name}] Baseline extraction failed: {error}")


threads = []
for groundtruth in GROUNDTRUTHS:
    thread = Thread(target=run_baseline_for_dataset, args=(groundtruth,))
    thread.start()
    threads.append(thread)

for thread in threads:
    thread.join()

print(f"Completed datasets: {len(generated_ll_objects)} / {len(GROUNDTRUTHS)}")
if baseline_errors:
    print("Baseline errors:", baseline_errors)

Actors STARTING... (attempt 1)
No feedback provided!
Actors DONE...
actors=[Actor(name='Citizens', description='Individuals who report issues and malfunctions in the urban environment.'), Actor(name='Municipal Operators', description='Staff members of the Municipality of Turin who review and manage citizen reports.'), Actor(name='External Maintenance Personnel', description='Workers from external companies who address specific issues reported by citizens.'), Actor(name='Municipal Administrators', description='Officials who configure the system and oversee the assignment of reports.'), Actor(name='Non-registered Users', description='Individuals who can view reports and statistics without needing to register.')]
High Level Goals STARTING... (attempt 1)
No feedback provided!
This is the provided sys prompt:  You are a helpful assistant expert in software engineering tasks.You're tasked to extract high level goals from a software description for each provided actor that is expected to inte

## 3. Proposed iterative bottom-up extension

The extension starts from the evaluated top-down hierarchy. For each iteration, `run_global_goal_cycle` reconstructs a candidate high-level goal from every branch, evaluates it globally, and updates only the parts of the hierarchy that require revision.

The cycle also performs a global documentation-coverage check after all current branches are confirmed. When the evaluator identifies a functional intention that is present in the documentation but absent from the current high-level-goal collection, the new high-level goal is appended. The pipeline then returns to the preceding top-down step: low-level goals are generated and evaluated for the new high-level goal. After this selective top-down generation, the bottom-up reconstruction and global evaluation are executed again on the updated hierarchy.

Existing confirmed branches are preserved; only revised branches and newly added high-level goals receive a new low-level decomposition.

### 3.1 Complete refinement–abstraction–verification cycle

The cycle receives the definitive baseline `HighLevelGoals` and `LowLevelGoals` objects. The same reflection-based top-down low-level generator used in the baseline is injected as a callback.

At each iteration the orchestrator:

1. reconstructs each branch bottom-up from its current low-level goals;
2. evaluates the reconstructed goal against its parent, all existing high-level goals, and the documentation;
3. updates the high-level-goal collection when required;
4. selectively regenerates and evaluates low-level goals for changed or newly introduced high-level goals;
5. starts a new bottom-up iteration using the updated hierarchy;
6. once all branches are confirmed, checks whether the documentation still contains missing high-level goals.

Each iteration is persisted through `GlobalGoalCycleIteration`, including traceability, decisions, added goals, regenerated decompositions, errors, convergence status, and the deterministic state signature.

In [ ]:
from src.bottom_up.goal_cycle_orchestrator import run_global_goal_cycle

MAX_GLOBAL_CYCLE_ITERATIONS = 2

global_cycle_results = {}
global_cycle_errors = {}
final_high_level_goals_objects = {}
final_low_level_goals_objects = {}

for dataset_name, initial_high_level_goals in generated_hl_objects.items():
    if dataset_name not in generated_ll_objects:
        global_cycle_errors[dataset_name] = (
            "Initial low-level goals are not available."
        )
        print(
            f"[{dataset_name}] Global goal cycle skipped: "
            "initial low-level goals are not available."
        )
        continue

    initial_low_level_goals = generated_ll_objects[dataset_name]
    description = generated_descriptions[dataset_name]
    actors = generated_actor_objects[dataset_name]

    def regenerate_low_level_goals_for_cycle(
        high_level_goals,
        _description=description,
        _actors=actors,
    ):
        regenerated, _score, _critique = (
            generate_evaluated_low_level_goals(
                high_level_goals=high_level_goals,
                description=_description,
                actors=_actors,
            )
        )
        return regenerated

    try:
        cycle_result = run_global_goal_cycle(
            project_description=description,
            initial_high_level_goals=initial_high_level_goals,
            initial_low_level_goals=initial_low_level_goals,
            regenerate_low_level_goals=(
                regenerate_low_level_goals_for_cycle
            ),
            max_iterations=MAX_GLOBAL_CYCLE_ITERATIONS,
        )
    except Exception as error:
        global_cycle_errors[dataset_name] = (
            f"{type(error).__name__}: {error}"
        )
        print(
            f"[{dataset_name}] Global goal cycle failed: "
            f"{type(error).__name__}: {error}"
        )
        continue

    global_cycle_results[dataset_name] = cycle_result
    final_high_level_goals_objects[dataset_name] = (
        cycle_result.final_high_level_goals
    )
    final_low_level_goals_objects[dataset_name] = (
        cycle_result.final_low_level_goals
    )

    output_file = output_file_for(dataset_name)
    with output_file.open("r", encoding="utf-8") as file:
        output = json.load(file)

    output["globalGoalCycle"] = cycle_result.model_dump(
        mode="json"
    )
    output["finalHighLevelGoals"] = (
        cycle_result.final_high_level_goals.model_dump(
            mode="json"
        )
    )
    output["finalLowLevelGoals"] = (
        cycle_result.final_low_level_goals.model_dump(
            mode="json"
        )
    )

    with output_file.open("w", encoding="utf-8") as file:
        json.dump(
            output,
            file,
            indent=4,
            ensure_ascii=False,
        )

    print(
        f"[{dataset_name}] Global cycle completed: "
        f"converged={cycle_result.converged}, "
        f"stop_reason={cycle_result.stop_reason}, "
        f"iterations={cycle_result.completed_iterations}."
    )

SyntaxError: no binding for nonlocal 'regeneration_call_counter' found (2972704738.py, line 95)

### 3.2 Cycle result inspection

The following cell prints a compact per-dataset summary. Detailed iteration artifacts are already persisted under `globalGoalCycle` in the corresponding JSON output.

In [ ]:
for dataset_name, cycle_result in global_cycle_results.items():
    print()
    print(f"[{dataset_name}]")
    print(f"  Converged: {cycle_result.converged}")
    print(f"  Stop reason: {cycle_result.stop_reason}")
    print(
        f"  Completed iterations: "
        f"{cycle_result.completed_iterations}"
    )
    print(
        f"  Final high-level goals: "
        f"{len(cycle_result.final_high_level_goals.goals)}"
    )
    print(
        f"  Final low-level goals: "
        f"{len(cycle_result.final_low_level_goals.low_level_goals)}"
    )
    print(
        f"  High-level goals added during the cycle: "
        f"{len(cycle_result.added_high_level_goals)}"
    )
    for added_goal in cycle_result.added_high_level_goals:
        print(f"    + {added_goal.name}")

    if cycle_result.unresolved_bottom_up_errors:
        print(
            "  Unresolved bottom-up errors:",
            cycle_result.unresolved_bottom_up_errors,
        )

    if cycle_result.unresolved_global_evaluation_errors:
        print(
            "  Unresolved global-evaluation errors:",
            cycle_result.unresolved_global_evaluation_errors,
        )

    if cycle_result.unresolved_empty_branches:
        print(
            "  Unresolved empty branches:",
            cycle_result.unresolved_empty_branches,
        )

if global_cycle_errors:
    print()
    print("Global cycle execution errors:", global_cycle_errors)

### 3.3 Final goal collections

The final collections are taken directly from `GlobalGoalCycleResult`. They therefore include high-level goals added by the branch evaluator or by the documentation-coverage evaluator, together with the low-level goals generated through the same evaluated top-down mechanism. A newly added high-level goal is not considered final until its decomposition has entered a subsequent bottom-up reconstruction and global re-evaluation, unless the maximum-iteration bound is reached first.

In [ ]:
final_high_level_goals = {
    dataset_name: [
        goal.description
        for goal in goals.goals
    ]
    for dataset_name, goals in final_high_level_goals_objects.items()
}

final_low_level_goals = {
    dataset_name: [
        goal.description
        for goal in goals.low_level_goals
    ]
    for dataset_name, goals in final_low_level_goals_objects.items()
}

for dataset_name in final_low_level_goals_objects:
    print(
        f"[{dataset_name}] Final goal collections available: "
        f"{len(final_high_level_goals_objects[dataset_name].goals)} HLG, "
        f"{len(final_low_level_goals_objects[dataset_name].low_level_goals)} LLG."
    )

## 4. Output persistence and traceability

Each dataset JSON contains the baseline artifacts and the complete iterative-extension result:

- `actors`;
- `highLevelGoals`;
- `lowLevelGoals`;
- `baselineScores`;
- `baselineCritiques`;
- `globalGoalCycle`;
- `finalHighLevelGoals`;
- `finalLowLevelGoals`.

`globalGoalCycle` contains the convergence status, stop reason, final collections, unresolved errors, and the complete ordered trace of all iterations. The traceability metadata is persisted for analysis but is never exposed to the bottom-up generator.

In [ ]:
for groundtruth in GROUNDTRUTHS:
    dataset_name = groundtruth["name"]
    output_file = output_file_for(dataset_name)

    if not output_file.exists():
        print(f"[{dataset_name}] Output file not available.")
        continue

    with output_file.open("r", encoding="utf-8") as file:
        output = json.load(file)

    print(
        f"[{dataset_name}] Saved sections: "
        f"{', '.join(output.keys())}"
    )


## 5. Goal-to-API alignment

The API alignment stage remains part of the original architecture. It is executed after the iterative extension and consumes `finalLowLevelGoals`, namely the last fully evaluated low-level decomposition returned by the cycle.

### 5.1 API extraction from Swagger

For each dataset that provides a Swagger source, this block extracts the available APIs.


In [ ]:
api_lists = {}

for groundtruth in GROUNDTRUTHS:
    dataset_name = groundtruth["name"]

    if "swagger" not in groundtruth:
        print(f"[{dataset_name}] No Swagger source configured; skipping API extraction.")
        continue

    print(f"[{dataset_name}] API extraction started...")
    api_lists[dataset_name] = get_api_list_from_swagger(
        link=groundtruth["swagger"]
    )
    print(
        f"[{dataset_name}] API extraction completed: "
        f"{len(api_lists[dataset_name])} API(s)."
    )


### 5.2 API mapping to final low-level goals

Each extracted API list is mapped to the final low-level goals returned by the global cycle. Datasets for which the cycle could not produce a final result are skipped and recorded separately.

In [ ]:
api_mappings = {}
api_mapping_skipped_due_to_cycle_errors = {}

for dataset_name, api_list in api_lists.items():
    if dataset_name not in final_low_level_goals_objects:
        error = global_cycle_errors.get(
            dataset_name,
            "Final low-level goals are not available.",
        )
        api_mapping_skipped_due_to_cycle_errors[dataset_name] = error
        print(f"[{dataset_name}] API mapping skipped: {error}")
        continue

    low_level_goals_for_mapping = (
        final_low_level_goals_objects[dataset_name]
    )

    print(f"[{dataset_name}] API mapping started...")
    mappings = generate_mapping_apis_goals(
        low_level_goals_for_mapping,
        api_list,
    )
    api_mappings[dataset_name] = mappings

    print_api_goal_mapping(mappings)

    mapping_file = OUTPUT_PATH / (
        f"final_mapping_{dataset_name}_{PROMPTING_MODE.name}"
        f"{'_noLlama' if LLAMA_ABLATION else ''}.json"
    )

    with mapping_file.open("w", encoding="utf-8") as file:
        json.dump(
            [mapping.model_dump(mode="json") for mapping in mappings],
            file,
            indent=4,
            ensure_ascii=False,
        )

    print(f"[{dataset_name}] API mapping saved to {mapping_file}.")

## 6. Execution summary

This notebook intentionally does not compute precision, recall, F1, semantic-similarity curves, or LLM-as-a-judge comparison statistics. Those analyses should read the persisted JSON files from a separate experimental-evaluation notebook, ensuring that generation and evaluation remain reproducible and independently repeatable.


In [ ]:
print("Execution summary")
print("-----------------")
print(f"Configured datasets: {len(GROUNDTRUTHS)}")
print(f"Baseline completed: {len(generated_ll_objects)}")
print(f"Global cycles completed: {len(global_cycle_results)}")
print(
    "Converged global cycles: "
    f"{sum(1 for result in global_cycle_results.values() if result.converged)}"
)
print(
    "Cycles stopped without convergence: "
    f"{sum(1 for result in global_cycle_results.values() if not result.converged)}"
)
print(f"Final goal collections available: {len(final_low_level_goals_objects)}")
print(f"API mappings completed: {len(api_mappings)}")
print(f"Global cycle execution errors: {len(global_cycle_errors)}")
print(
    "API mappings skipped due to cycle errors: "
    f"{len(api_mapping_skipped_due_to_cycle_errors)}"
)

if baseline_errors:
    print("Baseline errors:", baseline_errors)

if global_cycle_errors:
    print("Global cycle errors:", global_cycle_errors)

## 7. Standalone bottom-up run (baseline comparison)

This section runs **only** the new bottom-up extension, starting from a top-down baseline output already saved on disk under `output/`. It does **not** re-run the original actor, high-level-goal, or low-level-goal extraction pipeline.

The standalone cycle now includes the complete feedback loop implemented in the new architecture:
1. load an existing top-down baseline JSON;
2. deterministically map its flat LLGs to their existing HLG parents;
3. reconstruct one HLG candidate bottom-up for each branch;
4. evaluate each branch against its parent, the other HLGs, and the project documentation;
5. if the evaluator requests a new or replacement HLG, call the **original top-down HLG generator** through `generate_high_level_goals_from_request`;
6. add or replace the generated HLG deterministically in the current collection;
7. regenerate only the LLGs of branches that require a new decomposition;
8. repeat until all branches are confirmed and documentation coverage is complete, or until a stopping condition is reached.

The HLG generator is not told that it is correcting a previous result: the adapter forwards only the focused project description and actor contained in the evaluator request, with `feedback=None`.

Requirement: only section 1 (API keys) needs to have been run before this section.


Resource lifecycle: the standalone execution reinitializes its API clients at the start of the run, applies explicit request timeouts, and closes the OpenAI/Groq HTTP clients in a `finally` block when the run ends or raises an exception. The Jupyter kernel itself intentionally remains alive so that subsequent cells can still be executed.


In [9]:
from pathlib import Path
import gc
import json
import sys
import time

# Self-contained setup: this section can run against an already-saved top-down
# baseline without executing the previous pipeline sections.
STANDALONE_PROJECT_ROOT = Path.cwd()
if not (STANDALONE_PROJECT_ROOT / "src").exists() and (
    STANDALONE_PROJECT_ROOT.parent / "src"
).exists():
    STANDALONE_PROJECT_ROOT = STANDALONE_PROJECT_ROOT.parent

if str(STANDALONE_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_PROJECT_ROOT))

OUTPUT_PATH = STANDALONE_PROJECT_ROOT / "output"

# Existing top-down baseline JSON to use as input for the standalone
# bottom-up run. Change this to point at a different saved output file.
baseline_output_file = Path(
    "output/SIA Project 25 26_ONE_SHOT.json"
)
assert baseline_output_file.exists(), (
    f"Baseline file not found: {baseline_output_file}. "
    "Run section 2 first, or point baseline_output_file at an existing "
    "output/*.json file."
)

bottom_up_input_file = (
    OUTPUT_PATH
    / "bottom_up_inputs"
    / f"{baseline_output_file.stem}_bottom_up_input.json"
)
standalone_iterations_dir = (
    OUTPUT_PATH
    / "bottom_up_iterations"
    / baseline_output_file.stem
)

# Prevent a single remote request from keeping the notebook busy indefinitely.
# These values affect only the clients recreated for the standalone run.
STANDALONE_REQUEST_TIMEOUT_SECONDS = 120.0
STANDALONE_API_MAX_RETRIES = 1


def _close_client_safely(api_client):
    """Close one OpenAI-compatible client without masking pipeline errors."""
    if api_client is None:
        return
    try:
        api_client.close()
    except Exception as close_error:
        print(
            "Warning while closing an API client: "
            f"{type(close_error).__name__}: {close_error}"
        )


def reset_standalone_llm_clients():
    """Create fresh bounded-lifetime API clients for one standalone run.

    The functions imported by extractor/evaluator use the globals stored inside
    src.llm_clients, so replacing those globals is enough to make the complete
    bottom-up cycle use these fresh clients.
    """
    from openai import OpenAI
    from key import get_key_openai, get_key_llama
    import src.llm_clients as llm_clients

    # Close stale clients left by a previous notebook execution before creating
    # new ones. This also makes re-running the standalone cell safe.
    _close_client_safely(getattr(llm_clients, "client", None))
    _close_client_safely(getattr(llm_clients, "llama", None))

    llm_clients.client = OpenAI(
        api_key=get_key_openai(),
        timeout=STANDALONE_REQUEST_TIMEOUT_SECONDS,
        max_retries=STANDALONE_API_MAX_RETRIES,
    )
    llm_clients.llama = OpenAI(
        api_key=get_key_llama(),
        base_url="https://api.groq.com/openai/v1",
        timeout=STANDALONE_REQUEST_TIMEOUT_SECONDS,
        max_retries=STANDALONE_API_MAX_RETRIES,
    )


def close_standalone_llm_clients():
    """Release HTTP connections opened by the standalone LLM execution."""
    module = sys.modules.get("src.llm_clients")
    if module is not None:
        _close_client_safely(getattr(module, "client", None))
        _close_client_safely(getattr(module, "llama", None))
    gc.collect()


print("Baseline file:", baseline_output_file)
print("Bottom-up mapped input:", bottom_up_input_file)
print("Cycle trace directory:", standalone_iterations_dir)
print(
    "Standalone request timeout: "
    f"{STANDALONE_REQUEST_TIMEOUT_SECONDS:.0f}s per API request"
)


Baseline file: output\SIA Project 25 26_ONE_SHOT.json
Bottom-up mapped input: c:\Users\agnes_lryeu3v\Desktop\tesi\output\bottom_up_inputs\SIA Project 25 26_ONE_SHOT_bottom_up_input.json
Cycle trace directory: c:\Users\agnes_lryeu3v\Desktop\tesi\output\bottom_up_iterations\SIA Project 25 26_ONE_SHOT
Standalone request timeout: 120s per API request


In [10]:
from src.bottom_up.low_level_goal_mapper import (
    map_low_level_goals,
    load_mapped_bottom_up_input,
)

# Groups the existing (flat) low-level goals under their existing high-level
# parents, using the deterministic BRANCH_SPECIFICATIONS registered for this
# dataset/prompting-mode/ablation combination. No LLM call happens here.
mapped_payload = map_low_level_goals(
    source_file=baseline_output_file,
    destination_file=bottom_up_input_file,
)
standalone_source_metadata = mapped_payload["source"]

(
    standalone_project_description,
    standalone_initial_high_level_goals,
    standalone_initial_low_level_goals,
) = load_mapped_bottom_up_input(bottom_up_input_file)

print(f"Dataset: {standalone_source_metadata['dataset_name']}")
print(f"Prompting mode: {standalone_source_metadata['prompting_mode']}")
print(f"No-LLaMA ablation: {standalone_source_metadata['no_llama']}")
print(f"Initial high-level goals: {len(standalone_initial_high_level_goals.goals)}")
print(
    "Initial low-level goals: "
    f"{len(standalone_initial_low_level_goals.low_level_goals)}"
)


Dataset: SIA Project 25 26
Prompting mode: ONE_SHOT
No-LLaMA ablation: False
Initial high-level goals: 5
Initial low-level goals: 17


In [11]:
from src.data_model import Actors
from src.extraction.extractor import (
    generate_high_level_goals_from_request,
    generate_low_level_goals,
)
from src.self_critique.refine_response import (
    EvalMode,
    generate_response_with_reflection,
)
from src.examples.shot_learning import ShotPromptingMode

# Must match the prompting mode / ablation used to produce the baseline file
# selected above, so that every regenerated goal follows the same conventions.
STANDALONE_PROMPTING_MODE = ShotPromptingMode[
    standalone_source_metadata["prompting_mode"]
]
STANDALONE_LLAMA_ABLATION = standalone_source_metadata["no_llama"]

# The baseline JSON only stores actor names, not full Actor objects, so the
# actors are rebuilt here as the unique set already attached to the mapped
# high-level goals (in source order, without duplicates). These actors are used
# only by the existing low-level reflection/evaluation callback.
seen_actor_names = set()
standalone_actors_list = []
for goal in standalone_initial_high_level_goals.goals:
    if goal.actor.name not in seen_actor_names:
        seen_actor_names.add(goal.actor.name)
        standalone_actors_list.append(goal.actor)
standalone_actors = Actors(actors=standalone_actors_list)


def generate_high_level_goals_for_standalone_cycle(request):
    """HLG callback used by the new bottom-up cycle.

    The evaluator creates a HighLevelGoalGenerationRequest, but the original
    top-down generator receives only request.generator_input.project_description
    and request.generator_input.actors through the adapter. No previous HLG,
    branch id, evaluator decision, or correction metadata is exposed to it.
    """
    return generate_high_level_goals_from_request(
        request=request,
        mode=STANDALONE_PROMPTING_MODE,
    )


def regenerate_low_level_goals_for_standalone_cycle(high_level_goals):
    """Regenerate selected LLG branches with a compact reflection context.

    The generation input is unchanged. Only the evaluation message is reduced:
    it receives a short task description and only the actors associated with the
    HLGs currently selected by the orchestrator.
    """
    relevant_actors = Actors(
        actors=list({
            goal.actor.name: goal.actor
            for goal in high_level_goals.goals
        }.values())
    )

    compact_description = (
        "Generate and evaluate low-level goals that correctly operationalize "
        "the provided high-level goals, preserving their actor, functional "
        "intention, scope and responsibilities."
    )

    regenerated, _score, _critique = generate_response_with_reflection(
        "Low Level Goals",
        generate_low_level_goals,
        define_args=(high_level_goals,),
        eval_mode=EvalMode.LOW_LEVEL,
        eval_args=(
            compact_description,
            relevant_actors,
            high_level_goals,
        ),
        shotPromptingMode=STANDALONE_PROMPTING_MODE,
        llama_ablation=STANDALONE_LLAMA_ABLATION,
    )
    return regenerated


In [12]:
from src.bottom_up.goal_cycle_orchestrator import run_global_goal_cycle

# More than two iterations are useful because an HLG generated in one
# iteration must receive LLGs and then be reconstructed/evaluated again in a
# subsequent iteration before the cycle can converge.
STANDALONE_MAX_ITERATIONS = 5

# Each execution gets fresh API clients. The finally block guarantees that the
# HTTP connection pools are closed both after a successful run and after an
# exception, so the cell does not intentionally leave external resources open.
reset_standalone_llm_clients()
standalone_cycle_started_at = time.perf_counter()
print("Standalone bottom-up cycle started...")

try:
    standalone_cycle_result = run_global_goal_cycle(
        project_description=standalone_project_description,
        initial_high_level_goals=standalone_initial_high_level_goals,
        initial_low_level_goals=standalone_initial_low_level_goals,
        generate_high_level_goals=generate_high_level_goals_for_standalone_cycle,
        regenerate_low_level_goals=regenerate_low_level_goals_for_standalone_cycle,
        evaluation_output_directory=standalone_iterations_dir,
        max_iterations=STANDALONE_MAX_ITERATIONS,
    )
finally:
    elapsed = time.perf_counter() - standalone_cycle_started_at
    close_standalone_llm_clients()
    print(f"Standalone LLM resources closed after {elapsed:.1f}s.")

print(f"Converged: {standalone_cycle_result.converged}")
print(f"Stop reason: {standalone_cycle_result.stop_reason}")
print(
    "Completed iterations: "
    f"{standalone_cycle_result.completed_iterations}/"
    f"{standalone_cycle_result.max_iterations}"
)
print(
    "High-level goals added during the cycle: "
    f"{len(standalone_cycle_result.added_high_level_goals)}"
)
for added_goal in standalone_cycle_result.added_high_level_goals:
    print(f"  + [{added_goal.actor.name}] {added_goal.name}: {added_goal.description}")

if standalone_cycle_result.unresolved_bottom_up_errors:
    print(
        "Unresolved bottom-up errors:",
        standalone_cycle_result.unresolved_bottom_up_errors,
    )
if standalone_cycle_result.unresolved_global_evaluation_errors:
    print(
        "Unresolved global-evaluation errors:",
        standalone_cycle_result.unresolved_global_evaluation_errors,
    )
if standalone_cycle_result.unresolved_empty_branches:
    print(
        "Unresolved empty branches:",
        standalone_cycle_result.unresolved_empty_branches,
    )
if standalone_cycle_result.unresolved_documentation_coverage_error:
    print(
        "Unresolved documentation-coverage error:",
        standalone_cycle_result.unresolved_documentation_coverage_error,
    )
if standalone_cycle_result.unresolved_high_level_regeneration_error:
    print(
        "Unresolved high-level regeneration error:",
        standalone_cycle_result.unresolved_high_level_regeneration_error,
    )


Standalone bottom-up cycle started...
No feedback provided!
This is the provided sys prompt:  You are a helpful assistant expert in software engineering tasks.You're tasked to extract high level goals from a software description for each provided actor that is expected to interact with the software.Following the Goal-Oriented Requirements Engineering (GORE) frameworks, high-level goals are strategic objectives that define the 'why' behind a system. They are usually abstract, business-oriented, and independent of technical implementation. They represent the needs of stakeholders or the organization. Focus: Vision and justification. Generate ONLY the functional goals.
Low Level Goals STARTING... (attempt 1)
No feedback provided!
This is the provided sys prompt:  You are a helpful assistant expert in software engineering tasks. Elicit low-level goals for a specific stakeholder in a software project. The low-level goals that you create MUST be structured to match against a set of API calls

In [13]:
print("=== Comparison: top-down baseline vs. after the bottom-up cycle ===\n")

print(
    "High-level goals — baseline: "
    f"{len(standalone_initial_high_level_goals.goals)} | "
    f"after bottom-up: {len(standalone_cycle_result.final_high_level_goals.goals)}"
)
print(
    "Low-level goals  — baseline: "
    f"{len(standalone_initial_low_level_goals.low_level_goals)} | "
    f"after bottom-up: {len(standalone_cycle_result.final_low_level_goals.low_level_goals)}\n"
)

print("--- High-level goals: baseline (top-down only) ---")
for goal in standalone_initial_high_level_goals.goals:
    print(f"- [{goal.actor.name}] {goal.name}: {goal.description}")

print("\n--- High-level goals: after the bottom-up cycle ---")
for goal in standalone_cycle_result.final_high_level_goals.goals:
    print(f"- [{goal.actor.name}] {goal.name}: {goal.description}")

print("\n=== Iteration trace ===")
for iteration in standalone_cycle_result.iterations:
    print(f"\n--- Iteration {iteration.iteration} ---")

    if iteration.global_evaluations:
        print("Branch decisions:")
        for branch_id, evaluation in iteration.global_evaluations.items():
            print(
                f"- {branch_id}: {evaluation.decision} "
                f"— parent: {evaluation.original_high_level_goal.name}"
            )

    if iteration.high_level_generation_requests:
        print("HLG generation requests emitted by the evaluator:")
        for request in iteration.high_level_generation_requests:
            print(
                f"- {request.request_id}: {request.action} "
                f"(source={request.source})"
            )
            print(
                "  generator project description: "
                f"{request.generator_input.project_description}"
            )
            print(
                "  generator actors: "
                + ", ".join(
                    actor.name for actor in request.generator_input.actors.actors
                )
            )

    if iteration.generated_high_level_goals is not None:
        print("HLGs generated by the original top-down generator:")
        for goal in iteration.generated_high_level_goals.goals:
            print(f"  + [{goal.actor.name}] {goal.name}: {goal.description}")

    if iteration.newly_added_high_level_goals:
        print("HLGs appended to the collection in this iteration:")
        for goal in iteration.newly_added_high_level_goals:
            print(f"  + [{goal.actor.name}] {goal.name}: {goal.description}")

    if iteration.documentation_coverage is not None:
        print(
            "Documentation coverage: "
            f"{iteration.documentation_coverage.status}"
        )

# Summary of the last executed iteration.
last_iteration = standalone_cycle_result.iterations[-1]
decision_counts = {}
for evaluation in last_iteration.global_evaluations.values():
    decision_counts[evaluation.decision] = (
        decision_counts.get(evaluation.decision, 0) + 1
    )
print("\nDecision counts in last iteration:", decision_counts)


=== Comparison: top-down baseline vs. after the bottom-up cycle ===

High-level goals — baseline: 5 | after bottom-up: 17
Low-level goals  — baseline: 17 | after bottom-up: 49

--- High-level goals: baseline (top-down only) ---
- [Citizen] HLG_001: The citizen aims to easily report urban issues and inconveniences in their environment, ensuring their concerns are addressed by the municipality and can track the status of their reports throughout the resolution process, including receiving updates on changes to their reports over time.
- [Municipal Operator] HLG_002: The municipal operator seeks to efficiently review, manage, and respond to citizen reports to ensure timely resolutions of urban issues, while providing clear communication to citizens regarding the status of their reports and maintaining data accuracy and transparency.
- [External Maintenance Personnel] HLG_003: The external maintenance personnel aim to receive timely notifications for assigned reports and communicate effect

### 7.5 How to read this comparison

The standalone run now tests the complete implemented feedback loop while leaving the original top-down pipeline untouched.

- `CONFIRM_BRANCH`: the reconstructed bottom-up intention preserves the current HLG, so that branch is kept unchanged.
- `REGENERATE_LOW_LEVEL_GOALS`: the current HLG remains valid, but its present LLG decomposition is considered incomplete or semantically distorted; only its LLGs are regenerated.
- `MATCHES_OTHER_HIGH_LEVEL_GOAL`: the reconstructed intention corresponds more closely to another existing HLG; the current parent is kept, but its decomposition is regenerated and the mismatch remains traceable in the evaluator result.
- `ADD_NEW_HIGH_LEVEL_GOAL`: the evaluator identifies an autonomous documented intention not represented by the current HLG set. It creates a normal `HighLevelGoalGenerationRequest`; the original top-down HLG generator is called through the adapter, and the resulting HLG is appended by the orchestrator.
- `REWRITE_ORIGINAL_HIGH_LEVEL_GOAL`: the evaluator determines that the current HLG itself is defective with respect to the documentation. It creates a generation request for the correct documented intention; the original top-down HLG generator creates the replacement, and the orchestrator substitutes it in the collection.

After every HLG addition or replacement, the orchestrator generates the corresponding LLGs and the next iteration reconstructs and evaluates the updated branches again. Documentation coverage is checked only when all current branches are confirmed; any missing documented intention found there is also sent through the same normal top-down HLG-generation path.


In [14]:
standalone_output_file = (
    OUTPUT_PATH
    / f"{baseline_output_file.stem}_bottom_up_standalone.json"
)

with standalone_output_file.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "baseline_source_file": baseline_output_file.name,
            "bottom_up_input_file": str(bottom_up_input_file),
            "evaluation_output_directory": str(standalone_iterations_dir),
            "globalGoalCycle": standalone_cycle_result.model_dump(mode="json"),
            "finalHighLevelGoals": (
                standalone_cycle_result.final_high_level_goals.model_dump(
                    mode="json"
                )
            ),
            "finalLowLevelGoals": (
                standalone_cycle_result.final_low_level_goals.model_dump(
                    mode="json"
                )
            ),
        },
        file,
        indent=4,
        ensure_ascii=False,
    )

# Defensive no-op in the normal case; guarantees that a separately re-run save
# cell does not leave a previously recreated client open.
close_standalone_llm_clients()
print(f"Standalone bottom-up result saved to: {standalone_output_file}")
print("Standalone external API resources are closed; the Jupyter kernel remains ready/idle.")


Standalone bottom-up result saved to: c:\Users\agnes_lryeu3v\Desktop\tesi\output\SIA Project 25 26_ONE_SHOT_bottom_up_standalone.json
Standalone external API resources are closed; the Jupyter kernel remains ready/idle.
